# LLM (Large Language Model)
방대한 텍스트로 **다음에 올 단어(토큰)를 예측**하도록 훈련한 모델.  
> token: LLM이 글을 나눈 조각, 입출력의 단위

* 얼마나 큰지 파라미터 확인

In [ ]:
#!pip install -q transformers             # 코랩 기본 미설치 라이브러리(딥러닝 모델 허브 도구)
from transformers import AutoModel       # AutoModel: 이름만 주면 알맞은 모델 구조를 자동으로 불러옴

# 한국어 인코더(BERT)와 한국어 디코더(GPT) 두 종을 비교 — 계열은 달라도 '파라미터 수'로 규모를 잰다
for name in ['klue/bert-base', 'skt/kogpt2-base-v2']:   # 모델명은 Hugging Face 허브 기준(다음 절에서 소개)
    m = AutoModel.from_pretrained(name)                  # 사전학습된 가중치를 내려받아 메모리에 올림
    n = sum(p.numel() for p in m.parameters())           # 모든 가중치(파라미터) 개수의 합 = 모델 규모
    print(f'{name}: {n/1e6:.1f}M 파라미터')               # 1e6=100만 단위(M)로 환산해 보기 쉽게 출력

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


skt/kogpt2-base-v2: 125.2M 파라미터


klue/bert-base: 110.6M 파라미터   
skt/kogpt2-base-v2: 125.2M 파라미터

# 규칙기반 응답 vs LLM 응답 비교
LLM은 키워드가 아니라 문장의 의미를 본다. 처음 보는 표현에도 자연스러운 답을 만든다.  

[LLM의 특징]
* 수 많은 글에서 익힌 패턴으로 여기서 **가장 자연스러운 말**을 만들어낸다.  
(비유: 박사는 아니지만 책을 엄청 많이 읽어서 그럴듯 하게 말할 수 있는 사람)
* 이 질문 다음에 올 가장 그럴 듯한 글을 이어쓴다.  
(비유: 빈칸 이어쓰기 게임, 빈칸을 계속 이어서 채우는 게임)

### 0. 준비

In [1]:
from google.colab import drive
drive.mount('/content/drive')          # 구글 드라이브를 코랩에 연결(데이터 파일을 읽기 위해)

from pathlib import Path
import os, pandas as pd

# 드라이브의 gen-ai 폴더를 데이터 루트로 지정. DATA = 그 안의 data 폴더
ROOT = Path('/content/drive/MyDrive/kt cloud tech up/gen-ai')
DATA = ROOT / 'data'

from google.colab import userdata
for _n in ('GEMINI_API_KEY', 'GOOGLE_API_KEY'):
    try:
        os.environ['GEMINI_API_KEY'] = userdata.get(_n); break   # 키->환경변수
    except Exception:
        pass
from google import genai

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

Mounted at /content/drive


### 1. 데이터 로드: 상품문의 데이터

In [2]:
df = pd.read_csv(DATA / 'product_inquiries.csv')   # CSV를 표(DataFrame)로 읽는다
df[['inquiry_topic', 'inquiry_text']].head()        # 두 열만 골라 앞 5행 미리보기(.head())

,inquiry_topic,inquiry_text
0,규격,속건 마이크로화이버 타월 3매 실제 사이즈(가로세로높이)가 어떻게 되나요? 설치 공...
1,소재,스탠다드 후드 집업 소재가 비치는 편인가요? 안에 이너 받쳐 입어야 할까요?
2,내구성,오래 써도 녹슬거나 휘지 않나요? 내구성이 어떤지 궁금해요.
3,배송,지금 주문하면 주말 전에 받아볼 수 있을까요?
4,색상,네이비 색상 실물도 화면이랑 비슷한가요? 너무 쨍한 건 아닌지 궁금해요.


### 2. 응답 비교

2-1. 규칙기반   
: 사람이 직접 if/else로 '키워드'를 보고 답을 고른다(LLM 없이)

In [3]:
def rule_based_answer(text):
    if '배송' in text or '언제' in text:       # 문의에 이 단어가 들어 있으면
        return '배송은 결제 후 1~2일 내 출고됩니다.'
    elif '재입고' in text or '품절' in text:
        return '재입고 알림을 신청해 주세요.'
    elif '사이즈' in text or '크기' in text:
        return '상세페이지의 사이즈표를 확인해 주세요.'
    else:
        # 등록한 키워드가 하나도 없으면 이 기본 문구로 빠진다 → 규칙기반의 한계(자주 여기로)
        return '문의 감사합니다. 상담사가 확인 후 답변드리겠습니다.'

2-2. llm 기반   
: 규칙 대신 LLM에게 '역할(상담사) + 요청(정중히 한 문장)'을 주고 문의를 통째로 맡긴다

In [4]:
def llm_answer(text):
    prompt = f'너는 쇼핑몰 고객센터 상담사야. 다음 문의에 정중히 한 문장으로 답해줘.\n문의: {text}'
    return client.models.generate_content(model='gemini-2.5-flash-lite', contents=prompt).text

2-3. 비교

In [6]:
for t in df['inquiry_text'].head(3):    # 앞 3개 문의로 두 방식을 나란히 비교
    print('Q:', t)
    print('규칙:', rule_based_answer(t))
    print('LLM :', llm_answer(t), '\n')

Q: 속건 마이크로화이버 타월 3매 실제 사이즈(가로세로높이)가 어떻게 되나요? 설치 공간을 재보려고요.
규칙: 상세페이지의 사이즈표를 확인해 주세요.
LLM : 안녕하세요, 고객님. 속건 마이크로화이버 타월 3매의 실제 사이즈는 각 타월당 가로 40cm, 세로 80cm입니다. 

Q: 스탠다드 후드 집업 소재가 비치는 편인가요? 안에 이너 받쳐 입어야 할까요?
규칙: 문의 감사합니다. 상담사가 확인 후 답변드리겠습니다.
LLM : 스탠다드 후드 집업은 소재가 얇은 편이 아니라서 안에 이너를 꼭 받쳐 입으실 필요는 없지만, 개인의 취향에 따라 얇은 티셔츠 등을 함께 코디하시면 더욱 편안하게 착용하실 수 있습니다. 

Q: 오래 써도 녹슬거나 휘지 않나요? 내구성이 어떤지 궁금해요.
규칙: 문의 감사합니다. 상담사가 확인 후 답변드리겠습니다.
LLM : 고객님, 저희 제품은 튼튼한 소재로 제작되어 오래 사용하셔도 녹슬거나 휘어짐 없이 뛰어난 내구성을 자랑합니다. 



-> LLM이 일반 지식 생성에는 강하지만, 실질적으로 현업에 사용하려면 다음 과정이 필요하다.
* **도구 호출**로, **실시간 정보**(금일 재고 및 주문상태)
* **RAG**로, **사내 문서 지식**(환불 규정)
* **fine-tuning**으로, **회사 기준 분류**